# 03 Scoring, H3 Aggregation, and Web GeoJSON Export

This notebook follows the H3 example notebook's structure: H3 metadata first, then point/cell indexing, hierarchy checks, aggregation, neighborhood rings, and final deck.gl-ready export.


# Setup


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / ".deps"))
sys.path.insert(0, str(ROOT))

import geopandas as gpd
import h3

from scripts.build_outputs import (
    H3_RESOLUTION,
    BASELINE_WEIGHTS,
    TRACK_WEIGHTS,
    aggregate_to_h3,
    main,
)


# I. H3 preliminaries


In [ ]:
h3_meta = []
for res in range(5, 11):
    h3_meta.append({
        "resolution": res,
        "avg_edge_m": h3.average_hexagon_edge_length(res, unit="m"),
        "avg_area_m2": h3.average_hexagon_area(res, unit="m^2"),
        "avg_area_km2": h3.average_hexagon_area(res, unit="km^2"),
    })

pd.DataFrame(h3_meta).round(3)


In [ ]:
shanghai_center = (31.2304, 121.4737)
center_cells = []
for res in range(5, 11):
    cell = h3.latlng_to_cell(shanghai_center[0], shanghai_center[1], res)
    center_cells.append({
        "resolution": res,
        "h3_id": cell,
        "parent_res_5": h3.cell_to_parent(cell, 5) if res >= 5 else None,
    })

pd.DataFrame(center_cells)


In [ ]:
center_h3 = h3.latlng_to_cell(shanghai_center[0], shanghai_center[1], H3_RESOLUTION)
ring_rows = []
for k in range(0, 4):
    ring_rows.append({"k_ring": k, "cells": len(h3.grid_disk(center_h3, k))})
pd.DataFrame(ring_rows)


## I.1. Parent-child hierarchy


In [ ]:
parent = h3.cell_to_parent(center_h3, H3_RESOLUTION - 1)
children = h3.cell_to_children(parent, H3_RESOLUTION)

pd.Series({
    "center_h3_res8": center_h3,
    "parent_res7": parent,
    "children_at_res8": len(children),
    "center_is_child": center_h3 in children,
})


# II. Scoring design


In [ ]:
weights = pd.DataFrame({
    "baseline_weight": pd.Series(BASELINE_WEIGHTS),
    "track_a_weight": pd.Series(TRACK_WEIGHTS),
}).fillna("")
weights


In [ ]:
FORMULA = "\n".join([
    "For each 500 m grid cell:",
    "  category_score = 0.68 * distance_score + 0.32 * count_score",
    "  baseline_score = weighted mean of six everyday-service categories",
    "  track_score = weighted mean of five Healthy Lifestyle & Sport categories",
    "  composite_score = 0.60 * baseline_score + 0.40 * track_score",
    "",
    "For H3:",
    "  H3 score = mean of all 500 m grid cells whose centroids fall in the H3 cell",
])
print(FORMULA)


# III. Run or load the scored grid


In [ ]:
grid_path = ROOT / "data" / "processed" / "shanghai_500m_health_grid.parquet"
if not grid_path.exists():
    print("Grid cache missing; rebuilding all outputs from raw data.")
    main()
else:
    print(f"Using cached grid: {grid_path}")

grid = gpd.read_parquet(grid_path)
print(grid.shape)
grid[["grid_id", "district", "h3_id", "composite_score", "baseline_score", "track_score"]].head()


In [ ]:
score_fields = [
    "composite_score", "baseline_score", "track_score",
    "composite_walk", "composite_bike", "composite_transit", "composite_car",
]
grid[score_fields].describe().round(2)


# IV. Aggregate 500 m cells to H3 resolution 8


In [ ]:
h3_gdf = aggregate_to_h3(grid)
print(h3_gdf.shape)
h3_gdf[["h3_id", "district", "grid_count", "composite_score", "baseline_score", "track_score"]].head()


In [ ]:
pd.Series({
    "h3_cells": len(h3_gdf),
    "grid_cells": len(grid),
    "mean_grid_cells_per_h3": h3_gdf["grid_count"].mean(),
    "min_score": h3_gdf["composite_score"].min(),
    "mean_score": h3_gdf["composite_score"].mean(),
    "max_score": h3_gdf["composite_score"].max(),
}).round(2)


In [ ]:
district_summary = (
    h3_gdf.groupby("district")
    .agg(
        h3_cells=("h3_id", "count"),
        composite_mean=("composite_score", "mean"),
        health_mean=("track_score", "mean"),
        metro_dist_mean=("metro_dist_m", "mean"),
    )
    .sort_values("composite_mean", ascending=False)
    .round(2)
)
district_summary.head(20)


# V. H3 neighborhood analysis


In [ ]:
top_cell = h3_gdf.sort_values("composite_score", ascending=False).iloc[0]
neighbors = set(h3.grid_disk(top_cell["h3_id"], 1))
neighbor_df = h3_gdf[h3_gdf["h3_id"].isin(neighbors)].copy()

pd.Series({
    "top_h3": top_cell["h3_id"],
    "district": top_cell["district"],
    "top_score": top_cell["composite_score"],
    "neighbor_cells_found": len(neighbor_df),
    "neighbor_mean_score": neighbor_df["composite_score"].mean(),
}).round(2)


In [ ]:
top10 = h3_gdf.sort_values("composite_score", ascending=False)[[
    "h3_id", "district", "composite_score", "baseline_score", "track_score",
    "metro_dist_m", "rent_band", "top_amenities",
]].head(10)
top10


In [ ]:
bottom10 = h3_gdf.sort_values("composite_score", ascending=True)[[
    "h3_id", "district", "composite_score", "baseline_score", "track_score",
    "metro_dist_m", "rent_band", "top_amenities",
]].head(10)
bottom10


# VI. Optional lightweight visualization


In [ ]:
# This cell is optional. It runs only if matplotlib is installed.
try:
    import matplotlib.pyplot as plt
    ax = h3_gdf.plot(
        column="composite_score",
        cmap="YlGn",
        linewidth=0.05,
        edgecolor="white",
        figsize=(9, 9),
        legend=True,
    )
    ax.set_axis_off()
    ax.set_title("H3 resolution 8 composite score")
except Exception as exc:
    print("Static plot skipped:", exc)


# VII. Export full and web-ready GeoJSON


In [ ]:
output_dir = ROOT / "data" / "output"
app_data_dir = ROOT / "app" / "data"
output_dir.mkdir(parents=True, exist_ok=True)
app_data_dir.mkdir(parents=True, exist_ok=True)

full_geojson = output_dir / "shanghai_health_h3_r8.geojson"
web_geojson = app_data_dir / "shanghai_health_h3_r8.geojson"

h3_gdf.to_file(full_geojson, driver="GeoJSON")
print(full_geojson, full_geojson.stat().st_size)


In [ ]:
web_columns = [
    "h3_id", "district", "grid_count",
    "composite_score", "baseline_score", "track_score",
    "composite_walk", "composite_bike", "composite_transit", "composite_car",
    "baseline_walk", "baseline_bike", "baseline_transit", "baseline_car",
    "health_walk", "health_bike", "health_transit", "health_car",
    "food_walk_score", "healthcare_walk_score", "education_walk_score",
    "open_space_walk_score", "transit_access_walk_score", "civic_walk_score",
    "sport_walk_score", "green_outdoor_walk_score", "cycling_env_walk_score",
    "healthy_food_walk_score", "env_quality_walk_score",
    "food_walk_count", "healthcare_walk_count", "education_walk_count",
    "sport_walk_count", "green_outdoor_walk_count", "transit_access_walk_count",
    "metro_dist_m", "major_road_dist_m", "housing_price_proxy", "rent_band",
    "top_amenities", "geometry",
]

h3_web = h3_gdf[[c for c in web_columns if c in h3_gdf.columns]].copy()
h3_web.to_file(web_geojson, driver="GeoJSON")
print(web_geojson, web_geojson.stat().st_size)


In [ ]:
summary = {
    "project": "15-Minute Shanghai - Track A Healthy Lifestyle & Sport",
    "grid_size_m": 500,
    "h3_resolution": H3_RESOLUTION,
    "grid_cells": int(len(grid)),
    "h3_cells": int(len(h3_gdf)),
    "score_range": {
        "min": round(float(h3_gdf["composite_score"].min()), 2),
        "mean": round(float(h3_gdf["composite_score"].mean()), 2),
        "max": round(float(h3_gdf["composite_score"].max()), 2),
    },
    "method_notes": [
        "500 m cells are generated from a land and urban-activity mask so coastal districts, Changxing Island, and Chongming Island remain visible while open sea is excluded.",
        "The mask combines the 2020 built-up area, non-water land-use polygons, Baidu AOI polygons, and a small road-network buffer, clipped to the Shanghai administrative boundary.",
        "15-minute access is modelled with mode-specific distance catchments.",
        "Road data support cycling environment and major-road environmental context.",
    ],
}

summary_path = output_dir / "summary_notebook_export.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary


# VIII. deck.gl / H3HexagonLayer handoff


In [ ]:
DECK_GL_FIELDS = [
    "h3_id",
    "composite_walk", "composite_bike", "composite_transit", "composite_car",
    "baseline_walk", "health_walk",
    "sport_walk_score", "green_outdoor_walk_score", "cycling_env_walk_score",
    "metro_dist_m", "rent_band", "top_amenities",
]

h3_web[DECK_GL_FIELDS].head()


## Final export check

The web application loads `app/data/shanghai_health_h3_r8.geojson`. Each feature contains an H3 index, polygon geometry, mode-specific scores, baseline/Track A scores, detail-panel fields, and recommender fields. This mirrors the H3 example's final handoff to deck.gl, but uses Shanghai's health-and-sport scoring rather than Toulouse bus-stop counts.
